# Block-Wise FP8 / FP4 Quantization
*Compressing activations and weights to 8-bit and 4-bit without losing precision where it counts.*

---
**Component:** `learning/kernels/fp8_quantization/`
**Companion kernels:** `v0_naive.py` → `sm89_v1_act_quant.py` → `sm89_v2_fp8_gemm.py` → `sm89_v3_fp4_gemm.py`
**Source:** `kernel.py::act_quant`, `fp4_act_quant`, `fp8_gemm`, `fp4_gemm`


## Abstract

*Think of quantization as choosing a camera resolution: full-precision (FP32) is a 128-megapixel RAW file, FP16 is a 12-megapixel JPEG, FP8 is a 3-megapixel thumbnail, and FP4 is barely recognizable. The trick is that most of the information in a neural network lives in the "structure" of the activations, not in the fine decimal places — so you can shoot in thumbnail mode and still make out the scene. Block-wise quantization adapts the "exposure" (scale factor) independently for each small group of 128 or 32 values, so each neighborhood is optimally lit even if the global lighting varies widely. DSpark uses this to run its linear layers in FP8 and its speculative indexer in FP4, cutting memory bandwidth by 2–4× over BF16 with negligible quality loss.*

---

## Why Quantize?

Here's the fundamental bottleneck in LLM inference: reading weights from memory is far slower than computing with them. A single linear layer `y = x @ W` with a 4096×4096 weight matrix needs to stream $4096 \times 4096 \times 2$ bytes ≈ 32 MB from HBM for every token it generates. An H100's HBM bandwidth is ~3.4 TB/s — meaning each such layer takes about 10 microseconds of pure memory time, with the tensor cores sitting idle.

**The fix:** store weights in a smaller format so less data moves across the memory bus.

| Format | Bits/value | Bandwidth saving vs BF16 |
|--------|-----------|--------------------------|
| FP32 | 32 | 0.5× (uses more!) |
| BF16 | 16 | 1× (baseline) |
| FP8 | 8 | **2×** |
| FP4 | 4 | **4×** |

The savings are real — but only if you can quantize without destroying model quality. That's where block-wise scaling comes in.

In [ ]:
import math
print("setup complete")


## FP8 E4M3: A Compact Number Format

FP8 E4M3 packs a floating-point number into 8 bits using the same structure as FP32 — just much fewer bits for each field:

```
FP8 E4M3 bit layout (8 bits total):
┌─────┬─────────────────┬──────────────────────────┐
│  S  │   Exponent (4)  │       Mantissa (3)        │
└─────┴─────────────────┴──────────────────────────┘
  bit7      bits 6–3              bits 2–0
```

The rules follow IEEE 754 with one twist: NaN is encoded as `0xFF`, so there's no positive/negative NaN distinction. This gives a maximum representable value of **448.0**.

| Format | Max value | Distinct values |
|--------|-----------|-----------------|
| FP32 | ~3.4×10³⁸ | ~4 billion |
| BF16 | ~3.4×10³⁸ | 65,536 |
| FP8 E4M3 | **448.0** | **256** |

That tiny range is the crux of the problem. A weight matrix might have values spanning from –200 to +300 — no problem for FP8. But if a single outlier is 600, you can't represent it at all. Block-wise scaling solves this.

In [ ]:
# FP8 E4M3 constants (from kernel.py)
FP8_MAX = 448.0
FP8_MIN = -448.0

def fp8_encode(x):
    """Conceptually: clamp to fp8 range, then round to nearest fp8 value.
    In practice, 8 mantissa values per exponent band."""
    return max(FP8_MIN, min(FP8_MAX, x))

# What does FP8 precision look like near different magnitudes?
print("FP8 E4M3 precision at different scales:")
print(f"{'value':>10} {'fp8_max_err':>14} {'rel_error':>12}")
for exponent in range(-6, 9):
    val = 2.0 ** exponent
    # Within an exponent band, mantissa has 3 bits = 8 levels
    spacing = val / 8.0  # smallest representable increment
    rel = spacing / val
    if val <= FP8_MAX:
        print(f"{val:10.4f} {spacing:14.6f} {rel:12.4f} (1/{int(1/rel)})")

print(f"\nFP8 max representable: {FP8_MAX}")
print(f"Values above {FP8_MAX} saturate to NaN in E4M3")


## Block-Wise Quantization: Local Exposure Compensation

The idea is simple: instead of finding one global scale for the entire weight matrix, find a separate scale for each small block of 128 consecutive values. Each block is like a neighborhood with its own lighting — you expose for *that neighborhood*, not for the whole city.

**The algorithm for one block:**
1. Find the maximum absolute value in the block: $s = \max |x_i|$
2. Compute the scale: $\text{scale} = s / 448$  (so the largest value maps to ±448)
3. Quantize each value: $\hat{x}_i = \text{round}(x_i / \text{scale})$  then clip to FP8 range
4. Store the scale separately alongside the quantized values

**At inference time**, you dequantize on the fly: $x_i \approx \hat{x}_i \times \text{scale}$.

> **Block size matters.** Smaller blocks = more scales to store, but better local fidelity. DSpark uses `block_size = 128` for FP8 (weights/activations) and `block_size = 32` for FP4 (the indexer). The scale storage overhead is $1/\text{block\_size}$ extra — only 0.8% for FP8.

In [ ]:
import random
random.seed(42)

FP8_MAX = 448.0
block_size = 128

def fp8_quantize_block(block):
    """Quantize a block of floats to FP8 with a per-block scale."""
    amax = max(abs(x) for x in block)
    amax = max(amax, 1e-4)  # floor to avoid div-by-zero
    scale = amax / FP8_MAX
    quantized = [max(-FP8_MAX, min(FP8_MAX, x / scale)) for x in block]
    # 'store as FP8' conceptually means 3 mantissa bits of precision
    # We approximate this by rounding to 1/8 of the exponent band
    def round_fp8(v):
        if v == 0: return 0.0
        exp = math.floor(math.log2(abs(v)))
        spacing = 2**exp / 8.0
        return round(v / spacing) * spacing
    quantized_fp8 = [round_fp8(q) for q in quantized]
    return quantized_fp8, scale

# Simulate a block of activations
block = [random.gauss(0, 1.5) for _ in range(10)]
block[3] = 7.5  # mild outlier
block_q, scale = fp8_quantize_block(block)
dequant = [q * scale for q in block_q]

print("Block-wise FP8 quantization (showing first 10 of 128 elements):")
print(f"  amax = {max(abs(x) for x in block):.4f}, scale = {scale:.6f}")
print(f"{'idx':>4} {'original':>10} {'÷scale':>10} {'fp8_val':>10} {'dequant':>10} {'error':>8}")
for i in range(10):
    err = abs(block[i] - dequant[i])
    print(f"{i:4d} {block[i]:10.4f} {block[i]/scale:10.4f} {block_q[i]:10.4f} {dequant[i]:10.4f} {err:8.5f}")

# Max relative error in this block
rel_errors = [abs(block[i] - dequant[i]) / (abs(block[i]) + 1e-8) for i in range(10)]
print(f"\nMax relative error: {max(rel_errors):.4f} ({max(rel_errors)*100:.2f}%)")


## MXFP: Microscaling with Power-of-2 Scales

DSpark's `act_quant` supports an optional `scale_fmt = "ue8m0"` mode — this is the Microscaling FP (MXFP) standard. The "ue8m0" format stores scales as 8-bit *unsigned exponents only* (no mantissa, no sign). In other words, every scale must be an exact power of 2.

**Why only powers of 2?** Because multiplying by a power of 2 is free on hardware — you just add to the exponent bits in the IEEE 754 representation. No multiply instruction needed. This makes dequantization nearly instantaneous.

**The fast log2 ceiling trick** (used in `fast_log2_ceil`): to find the smallest power of 2 that's ≥ $s$, you can work directly with the IEEE 754 bit representation:

```python
# View the float as an integer, right-shift the exponent bits out, subtract bias
def fast_log2_ceil(x):
    bits = float_to_int32(x)           # raw IEEE 754 bits
    exp  = (bits >> 23) & 0xFF         # extract 8-bit exponent
    mant = bits & 0x7FFFFF             # extract 23-bit mantissa
    log2 = exp - 127                   # remove bias
    if mant != 0: log2 += 1            # ceiling: round up if not already a power of 2
    return log2
```

No division, no logarithm function — just bit manipulation. Then `fast_pow2(log2)` reconstructs the scale by reversing the process.

In [ ]:
import struct

def fast_log2_ceil_py(x):
    """Python equivalent of the IEEE 754 trick in kernel.py."""
    if x <= 0:
        raise ValueError(f"x must be positive, got {x}")
    # ceil(log2(x))
    log2_floor = math.floor(math.log2(x))
    is_exact_power = (x == 2.0 ** log2_floor)
    return log2_floor if is_exact_power else log2_floor + 1

def fast_round_scale_py(amax, fp8_max=448.0):
    """Round scale up to nearest power of 2."""
    ratio = amax / fp8_max
    exponent = fast_log2_ceil_py(ratio) if ratio > 0 else 0
    return 2.0 ** exponent

# Compare standard vs MXFP scale for various amax values
test_amaxes = [0.1, 0.5, 1.0, 3.7, 10.0, 100.0, 350.0, 448.0]
print("Standard vs MXFP (power-of-2) scales:")
print(f"{'amax':>8} {'std_scale':>12} {'mxfp_scale':>12} {'ratio':>8} {'prec_loss'}")
for amax in test_amaxes:
    std = amax / 448.0
    mxfp = fast_round_scale_py(amax)
    ratio = mxfp / std
    # Precision loss: MXFP scale is >= std, so quantized range is smaller
    max_val_after = amax / mxfp
    print(f"{amax:8.1f} {std:12.6f} {mxfp:12.6f} {ratio:8.3f}  max_repr={max_val_after:.1f} (≤448 ✓)")

print("\nMXFP scale is always ≥ standard scale, ensuring no overflow.")
print("Trade-off: slight precision loss (up to 2×) in exchange for exact power-of-2 arithmetic.")


## FP8 GEMM: Fused Scaled Matrix Multiplication

The `fp8_gemm_kernel` computes $Y = X W^\top$ where both $X$ and $W$ are stored in FP8 with block-wise FP32 scales. The kernel fuses quantization into the GEMM itself:

$$Y[m, n] = \left(\sum_k \hat{X}[m, k] \cdot \hat{W}[n, k]\right) \times s_X[m, \lfloor k/128 \rfloor] \times s_W[n, \lfloor k/128 \rfloor]$$

**In plain English:** multiply the quantized integers together (fast!), then multiply by the appropriate pair of scales to recover the floating-point result.

The key insight is that each output element $Y[m, n]$ depends on scales $s_X[m, \cdot]$ and $s_W[n, \cdot]$ — one scale pair per 128-element block of the inner dimension $k$. These scales are small (just one FP32 per 128 values) and can be kept in registers or shared memory during the GEMM.

> **Real-world impact:** On an H100, FP8 Tensor Cores run at ~2× the throughput of BF16 Tensor Cores. Combined with the 2× bandwidth saving, FP8 GEMM can be 3–4× faster than a naively implemented BF16 equivalent for large matrices.

In [ ]:
def fp8_gemm_reference(A, B, scales_a, scales_b, block_size=4):
    """
    Reference implementation of block-scaled FP8 GEMM.
    A: [M, K], B: [N, K] (both conceptually in FP8 but stored as float for clarity)
    scales_a: [M, K//block_size], scales_b: [N//block_size, K//block_size]
    Returns C: [M, N]
    """
    M, K = len(A), len(A[0])
    N = len(B)
    C = [[0.0] * N for _ in range(M)]
    for i in range(M):
        for j in range(N):
            acc = 0.0
            for kb in range(K // block_size):
                # FP8 dot product over this block
                block_dot = sum(
                    A[i][kb * block_size + l] * B[j][kb * block_size + l]
                    for l in range(block_size)
                )
                # Apply per-block scales
                sa = scales_a[i][kb]
                sb = scales_b[j // block_size][kb]
                acc += block_dot * sa * sb
            C[i][j] = acc
    return C

# Small example: M=3, K=8, N=4, block_size=4 (normally 128 in hardware)
random.seed(3)
M, K, N, bs = 3, 8, 4, 4

# Simulate quantized values (conceptually in FP8 range)
A_q = [[random.uniform(-4, 4) for _ in range(K)] for _ in range(M)]
B_q = [[random.uniform(-4, 4) for _ in range(K)] for _ in range(N)]
scales_a = [[random.uniform(0.01, 0.1) for _ in range(K // bs)] for _ in range(M)]
scales_b = [[random.uniform(0.01, 0.1) for _ in range(K // bs)] for _ in range(N // bs + 1)]

C = fp8_gemm_reference(A_q, B_q, scales_a, scales_b, block_size=bs)

# Compare to unscaled gemm (what you'd get if scales were 1)
C_unscaled = [[sum(A_q[i][k] * B_q[j][k] for k in range(K)) for j in range(N)] for i in range(M)]

print("FP8 GEMM (C = A @ B.T with per-block scales):")
for i in range(M):
    print(f"  row {i}: {[round(C[i][j], 4) for j in range(N)]}")
print("\nUnscaled GEMM (for comparison):")
for i in range(M):
    print(f"  row {i}: {[round(C_unscaled[i][j], 4) for j in range(N)]}")
print("\n(Scaled values are much smaller — scales ~0.01-0.1 applied per block)")


## FP4 E2M1: Pushing the Limits of Compression

FP4 E2M1 compresses each value to just 4 bits — two for the exponent and one for the mantissa:

```
FP4 E2M1 bit layout (4 bits total):
┌─────┬─────────────┬──────────────────┐
│  S  │ Exponent (2)│   Mantissa (1)   │
└─────┴─────────────┴──────────────────┘
  bit3    bits 2–1         bit 0
```

With only 16 distinct values (0–15), FP4 can represent just a handful of magnitudes. The maximum representable value is only **6.0**. DSpark uses FP4 specifically for the Indexer's compressed KV — a context where exact precision matters less than the ability to rank positions by relevance.

| Format | Max value | Values | Block size |
|--------|-----------|--------|------------|
| FP8 E4M3 | 448.0 | 256 | 128 |
| FP4 E2M1 | **6.0** | **16** | **32** |

The smaller block size for FP4 (32 vs 128) compensates for the reduced precision — more frequent re-calibration makes up for having fewer representable values.

In [ ]:
FP4_MAX = 6.0
FP4_MIN = -6.0
# All representable FP4 E2M1 values (positive, then mirrored)
FP4_VALS = [0.0, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 6.0]

def fp4_quantize_block(block, block_size=4):
    """Quantize a block to FP4 with power-of-2 MXFP scale."""
    amax = max(abs(x) for x in block)
    amax = max(amax, 6 * (2 ** -126))  # floor (from kernel.py)
    scale = fast_round_scale_py(amax, fp8_max=FP4_MAX)
    # Quantize: clamp to FP4 range, then round to nearest FP4 value
    def round_fp4(v):
        v_scaled = max(-FP4_MAX, min(FP4_MAX, v / scale))
        sign = 1 if v_scaled >= 0 else -1
        best = min(FP4_VALS, key=lambda f: abs(abs(v_scaled) - f))
        return sign * best
    quantized = [round_fp4(x) for x in block]
    return quantized, scale

# Example: a block of 8 values
block = [0.8, -2.1, 0.3, 4.5, -1.7, 0.05, 3.2, -0.6]
q, scale = fp4_quantize_block(block)
dq = [v * scale for v in q]

print(f"FP4 quantization (scale={scale:.4f}):")
print(f"{'val':>8} {'÷scale':>8} {'fp4_val':>8} {'dequant':>8} {'error':>7}")
for i, (x, qi, dqi) in enumerate(zip(block, q, dq)):
    print(f"{x:8.3f} {x/scale:8.3f} {qi:8.3f} {dqi:8.3f} {abs(x-dqi):7.4f}")

print(f"\nFP4 representable values (positive): {FP4_VALS}")
print(f"Max relative error vs FP8: FP4 has 1 mantissa bit vs 3 → ~4× more quantization error")


## Inplace Quantization: Simulating QAT on the Forward Pass

In production, DSpark runs `act_quant(tensor, block_size, inplace=True)` — which quantizes and immediately *dequantizes* back to BF16, in place, in a single pass. The tensor ends up in BF16 but with values that have been rounded to the FP8 grid.

**Why would you ever do this?** It's a Quantization-Aware Training (QAT) technique. During training, the forward pass simulates quantization error — the model sees the rounded values and learns to be robust to them. The backward pass still flows through gradients normally (using straight-through estimators). At inference time, you can then actually store the weights in FP8 and the model won't be surprised.

In short: `act_quant(..., inplace=True)` is the training-time stand-in for "what this tensor would look like if it were actually stored in FP8 and read back."

> **Fused kernel benefit:** Doing quantize + dequantize in a single kernel pass avoids writing the FP8-quantized tensor to HBM and reading it back. The intermediate FP8 values live in registers and never touch global memory.